# 04.2 — Planning with Todos (`TodoListMiddleware`)

Learn how to give a Deep Agent a built-in todo list so it can plan multi-step work and update its progress live.

**What we'll build:**

1. Opt in to task planning using `TodoListMiddleware()` (adds the `write_todos` tool)
2. Stream node-by-node updates with `agent.stream(..., stream_mode="updates")`
3. Capture and print every revision of the agent's plan

> **Setup:** Make sure Ollama is running locally (defaults to `ollama:qwen3.5:2b`), or set `DEEPAGENTS_MODEL` in your `.env` file.


## Step 1: Imports & Model Setup

Load the core agent factory, the planning middleware (`TodoListMiddleware`), environment config, and `rich` for clean terminal output.


In [1]:
################################ Imports and Model Setup ################################
import os

from deepagents import create_deep_agent  # Core agent factory
from dotenv import load_dotenv  # Loads .env variables
from langchain.agents.middleware import TodoListMiddleware  # Adds the write_todos tool
from rich import print  # Colorized console output

# Load environment variables from .env (if present)
load_dotenv()

# Use DEEPAGENTS_MODEL if set; otherwise default to a small local Ollama model
MODEL = os.getenv("DEEPAGENTS_MODEL", "ollama:qwen3.5:2b")

## Step 2: Enable Planning with `TodoListMiddleware`

- In **deepagents 0.7**, task planning is **opt-in** (in 0.6 it was on by default). Keeping it opt-in keeps simple agents lean.
- Passing `middleware=[TodoListMiddleware()]` gives the agent the `write_todos` tool.
- The `system_prompt` instructs the model to create a plan first, update it as it works, and then return the final answer.


In [2]:
################################ Agent Construction ################################
# Build the agent and opt in to planning via TodoListMiddleware
agent = create_deep_agent(
    model=MODEL,
    middleware=[TodoListMiddleware()],  # Without this, the agent has no write_todos tool
    system_prompt=(
        "You are a careful planner. For any multi-step request, FIRST call "
        "write_todos to lay out the steps, update it as you go, then give the "
        "final answer in a separate message."  # Split the answer so the last message is text
    ),
)

## Step 3: Define a Multi-Step Task

A prompt with **three small, distinct steps** is the sweet spot for a demo: complex enough to trigger `write_todos`, yet fast to run live.


In [3]:
################################ Task Prompt ################################
# Give the agent three distinct steps so it has a reason to build a todo list
task = (
    "plan and Do these three things: 1) compute 12 * 12, 2) reverse the word 'agents', "
    "3) give me both results in one final sentence."
)

## Step 4: Stream Updates & Track the Plan Live

- `stream_mode="updates"` yields one chunk each time a graph node finishes (`model`, `tools`, etc.), so we can watch the plan evolve in real time.
- When `write_todos` runs, the updated list appears under the `"todos"` key in the node's state update.
- We save each version to `todo_snapshots` so we can print the full revision history in Step 5.


In [4]:
# %% Step 4: Stream the run and watch the todo list change
################################ Streaming the Run ################################
# stream_mode="updates" yields one chunk per node that just ran, so we can see
# each write_todos call as it lands instead of waiting for the final state.
print("[bold cyan]Streaming updates:[/bold cyan]")
todo_snapshots = []
final_state = None

# Stream the run so each node's state delta arrives as it completes
for chunk in agent.stream({"messages": [{"role": "user", "content": task}]}, stream_mode="updates"):
    # Unpack the single node name -> state update pair in this chunk
    for node_name, update in chunk.items():
        # Skip non-dict updates (e.g. interrupts) so the field reads below stay safe
        if not isinstance(update, dict):
            continue
        # Todos live in state under the "todos" key once the tool is used.
        if "todos" in update:
            todo_snapshots.append(update["todos"])
            print(f"  [magenta]{node_name}[/magenta] updated the plan ({len(update['todos'])} items)")
        # Log tool requests so the trace shows which node chose write_todos
        for message in update.get("messages", []):
            for call in getattr(message, "tool_calls", []) or []:
                print(f"  [magenta]{node_name}[/magenta] called [green]{call['name']}[/green]")
        # Keep the last update so it holds the latest messages state
        final_state = update

Streaming updates:

model called write_todos

model called write_todos

tools updated the plan (3 items)

model called write_todos

tools updated the plan (3 items)

model called write_todos

tools updated the plan (3 items)

model called write_todos

tools updated the plan (3 items)

### Step 5: Show how the plan evolved

- Key: `see code`.
- Note: Note: square brackets are rich markup, so we use plain symbols here.
- Run this cell, then check the output before moving on.


In [5]:
################################ Plan History ################################
# %% Step 5: Show how the plan evolved
print(f"\n[bold cyan]The plan was revised {len(todo_snapshots)} time(s):[/bold cyan]")
# Replay every captured revision so the progress of the plan is visible
for index, snapshot in enumerate(todo_snapshots, start=1):
    print(f"\n[bold]Revision {index}[/bold]")
    for todo in snapshot:
        # Note: square brackets are rich markup, so we use plain symbols here.
        marks = {"pending": "○", "in_progress": "◐", "completed": "●"}
        mark = marks.get(todo.get("status"), "○")  # Fall back to pending on an unknown status
        print(f"  {mark} {todo.get('content')}")

# Warn instead of printing nothing when the model skipped planning entirely
if not todo_snapshots:
    print("  [yellow]The model answered without planning — try a longer task.[/yellow]")

# %%

The plan was revised 4 time(s):

Revision 1

◐ Plan and compute 12 * 12

○ Reverse the word 'agents'

○ Combine results into one final sentence

Revision 2

... (7 outputs left out)

Revision 4

● Plan and compute 12 * 12

● Reverse the word 'agents'

● Combine results into one final sentence